# 🎨 Indie Comic Generator - T4 Optimized Single Notebook

## ⚡ Complete pipeline in one notebook - models cache between cells!

### Features:
- **T4 GPU Optimized**: 768x768 resolution, 25 inference steps
- **Model Caching**: SDXL loads once, reused for all pages
- **Memory Management**: Auto-cleanup every 3 pages
- **Complete Pipeline**: Extraction → Fusion → Generation → PDF

⚠️ **Select Runtime > Change runtime type > T4 GPU before starting**

---

## 📝 Configuration
Edit the values below for your story:

In [ ]:
# ============================================================
# EDIT THESE VALUES
# ============================================================
CHARACTER_NAME = "Spider-Man"      # Any character (Wolverine, Batman, etc.)
STORY_WORLD = "Cyberpunk 2077"     # Any setting (Harry Potter, Wuthering Heights, etc.)
NUM_PAGES = 5                      # Pages to generate (1-10)
USE_LORA = False                   # True = SDXL+LoRA (best), False = SDXL Base
EXTREME_QUALITY = False            # True = Extreme Quality (896x896, 40 steps, IP-Adapter, strict consistency)

# Advanced settings (usually leave as is)
IMG_WIDTH = 768                    # T4 optimized (512-1024)
IMG_HEIGHT = 768                   # T4 optimized (512-1024)
INFERENCE_STEPS = 25               # T4 optimized (15-40)

print(f"🎭 Character: {CHARACTER_NAME}")
print(f"🌍 World: {STORY_WORLD}")
print(f"📄 Pages: {NUM_PAGES}")
print(f"🎨 Model: {'SDXL + LoRA' if USE_LORA else 'SDXL Base'}")
print(f"📐 Resolution: {IMG_WIDTH}x{IMG_HEIGHT}")
print(f"⚡ Steps: {INFERENCE_STEPS}")
print(f"💎 Extreme Quality: {EXTREME_QUALITY}")

## 📦 Step 1: Install Dependencies
This will install all required packages (takes 2-3 minutes)

In [ ]:
!pip install -q diffusers transformers accelerate safetensors \
    langchain-ollama langchain-core pyyaml \
    opencv-python-headless pillow scikit-image \
    peft torchmetrics matplotlib pandas

import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 📂 Step 2: Clone Repository
Downloads the pipeline code

In [ ]:
import os, subprocess

# Detect if running in Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/indie_comic_pipeline"
    if not os.path.exists(REPO_DIR):
        print("Cloning repository in Colab...")
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Cyberpunk-San/Indie-Comic.git", REPO_DIR], check=True)
    else:
         print("Repository already exists")
    os.chdir(REPO_DIR)
else:
    # Local setup: find indie_comic_pipeline directory
    cwd = os.getcwd()
    if os.path.basename(cwd) == "indie_comic_pipeline":
        os.chdir(cwd)
    elif os.path.exists(os.path.join(cwd, "indie_comic_pipeline")):
        os.chdir(os.path.join(cwd, "indie_comic_pipeline"))
    print(f"Working directory: {os.getcwd()}")

# Create output directories
os.makedirs("outputs/fusion", exist_ok=True)
os.makedirs("outputs/comics", exist_ok=True)
os.makedirs("outputs/characters", exist_ok=True)

print("✅ Repository ready")

## ⚡ Step 3: Apply T4 Optimizations
Configures settings for optimal T4 performance

In [ ]:
import yaml

with open('config/settings.yaml', 'r') as f:
    settings = yaml.safe_load(f)

# Apply settings
settings['generation']['guidance_scale'] = 7.5
settings['generation']['seed'] = 42

# Model settings
settings['models']['sdxl']['device'] = 'cuda'
settings['models']['sdxl']['memory_optimization'] = True
settings['models']['lora']['adapter_scale'] = 0.8

if EXTREME_QUALITY:
    settings['generation']['default_size'] = {'width': 896, 'height': 896}
    settings['generation']['inference_steps'] = 40
    settings['t4_optimizations'] = {
        'enabled': True,
        'cpu_offload': True,
        'attention_slicing': True,
        'vae_slicing': True,
        'disable_ipadapter': False,
        'clear_cache_every_n_steps': 3
    }
    settings['models']['ipadapter'] = settings.get('models', {}).get('ipadapter', {})
    settings['models']['ipadapter']['enabled'] = True
    settings['consistency'] = {
        'enable_clip': True,
        'enable_dinov2': True,
        'enable_ssim': True,
        'enable_edge': True,
        'enable_color': True,
        'enable_style': True,
        'threshold': 0.75
    }
else:
    settings['generation']['default_size'] = {'width': IMG_WIDTH, 'height': IMG_HEIGHT}
    settings['generation']['inference_steps'] = INFERENCE_STEPS
    settings['t4_optimizations'] = {
        'enabled': True,
        'cpu_offload': True,
        'attention_slicing': True,
        'vae_slicing': True,
        'disable_ipadapter': True,
        'clear_cache_every_n_steps': 3
    }
    settings['models']['ipadapter'] = settings.get('models', {}).get('ipadapter', {})
    settings['models']['ipadapter']['enabled'] = False
    settings['consistency'] = {
        'enable_clip': False,
        'enable_dinov2': False,
        'enable_ssim': True,
        'enable_edge': True,
        'enable_color': False,
        'enable_style': True,
        'threshold': 0.55
    }

with open('config/settings.yaml', 'w') as f:
    yaml.dump(settings, f)

print(f"✅ Optimizations applied | ExtremeQuality={EXTREME_QUALITY}")
print(f"   Attention Slicing: Enabled")

## 🦙 Step 4: Start Ollama
Starts the local LLM server for story extraction

In [ ]:
import subprocess
import threading
import time
import socket

# Start Ollama server in background
def run_ollama():
    subprocess.run(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

thread = threading.Thread(target=run_ollama, daemon=True)
thread.start()
time.sleep(3)

# Check if Ollama is running
def check_ollama():
    try:
        sock = socket.create_connection(("localhost", 11434), timeout=1)
        sock.close()
        return True
    except:
        return False

if check_ollama():
    print("✅ Ollama server running")
else:
    print("⚠️ Ollama starting, please wait...")
    time.sleep(5)

# Pull Llama 3.2 model
print("📥 Pulling Llama 3.2 model (first time only)...")
!ollama pull llama3.2
print("✅ Ollama ready")

## 🎭 Step 5: Extract Character Personality
Uses LLM to analyze character psychology

In [ ]:
import subprocess
import sys

print(f"Analyzing character: {CHARACTER_NAME}")
result = subprocess.run(
    [sys.executable, "langchain_code/character_extractor.py", CHARACTER_NAME],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
else:
    print("✅ Character extraction complete")

## 🌍 Step 6: Extract Story Setting
Analyzes the story world and environment

In [ ]:
print(f"Analyzing world: {STORY_WORLD}")
result = subprocess.run(
    [sys.executable, "langchain_code/story_extractor.py", STORY_WORLD],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
else:
    print("✅ Story extraction complete")

In [ ]:
def show_gpu_memory():
    """Display current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        free = torch.cuda.get_device_properties(0).total_memory / 1024**3 - reserved
        print(f"💾 GPU Memory: {allocated:.2f}GB used, {reserved:.2f}GB reserved, {free:.2f}GB free")
        if reserved > 13:
            print("   ⚠️ High memory usage - consider restarting runtime")
    else:
        print("💻 No GPU detected")

def clear_gpu_memory():
    """Force clear GPU cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        import gc
        gc.collect()
        print("🧹 GPU cache cleared")

show_gpu_memory()

## 🎨 Step 7: Generate Comic Pages
**This is the main generation loop.**

⚠️ **First page takes 30-60 seconds (loading models)**
⚠️ **Subsequent pages take 8-10 seconds each (models cached)**

The model stays loaded in GPU memory between pages!

In [ ]:
import subprocess
import sys
import time

print("=" * 70)
print("🎬 STARTING COMIC GENERATION")
print("=" * 70)
print(f"Character: {CHARACTER_NAME}")
print(f"World: {STORY_WORLD}")
print(f"Pages: {NUM_PAGES}")
print(f"Model: {'SDXL + LoRA' if USE_LORA else 'SDXL Base'}")
print("=" * 70)

total_start = time.time()

for page in range(1, NUM_PAGES + 1):
    page_start = time.time()
    print(f"\n{'=' * 50}")
    print(f"📖 PAGE {page} OF {NUM_PAGES}")
    print(f"{'=' * 50}")
    
    # Step 1: Fusion Engine (creates storyboard)
    print(f"[1/3] Creating storyboard for page {page}...")
    result = subprocess.run(
        [sys.executable, "langchain_code/fusion_engine.py", "--page", str(page)],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        break
    
    # Step 2: Emotion Recognition
    print(f"[2/3] Analyzing emotions for page {page}...")
    result = subprocess.run(
        [sys.executable, "langchain_code/emotion_recognition_engine.py", "--page", str(page)],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        break
    
    # Step 3: Generate Panels (model caches after first page!)
    print(f"[3/3] Generating panels for page {page}...")
    
    if USE_LORA:
        # First page needs to load model, subsequent pages use cache
        if page == 1:
            print("   🔄 Loading SDXL + LoRA model (first time - 30-60 sec)...")
            # Run character generation first to load model
            subprocess.run([sys.executable, "lora_code/run_lora_pipeline.py"], capture_output=True)
        
        result = subprocess.run(
            [sys.executable, "lora_code/generate_panels.py", "--page", str(page)],
            capture_output=True,
            text=True
        )
    else:
        if page == 1:
            print("   🔄 Loading SDXL Base model (first time - 30-60 sec)...")
            subprocess.run([sys.executable, "sdxl_code/run_sdxl_pipeline.py"], capture_output=True)
        
        result = subprocess.run(
            [sys.executable, "sdxl_code/generate_panels.py", "--page", str(page)],
            capture_output=True,
            text=True
        )
    
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        break
    
    page_time = time.time() - page_start
    print(f"✅ Page {page} completed in {page_time:.1f} seconds")
    
    # Clear GPU memory every 3 pages
    if page % 3 == 0:
        clear_gpu_memory()
        show_gpu_memory()

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"✅ ALL {NUM_PAGES} PAGES COMPLETED!")
print(f"⏱️ Total time: {total_time:.1f} seconds")
print(f"{'=' * 70}")

## 🖼️ Step 8: View Generated Comics
Displays the compiled page layouts

In [ ]:
from IPython.display import Image, display
import glob

style = "sdxl_lora_grid" if USE_LORA else "sdxl_base_grid"

for page in range(1, NUM_PAGES + 1):
    grid_path = f"outputs/comics/page_{page}_layout_{style}.png"
    if os.path.exists(grid_path):
        print(f"\n📄 Page {page}:")
        display(Image(grid_path))
    else:
        # Try alternative naming
        alt_paths = glob.glob(f"outputs/comics/page_{page}_layout_*_grid.png")
        if alt_paths:
            print(f"\n📄 Page {page}:")
            display(Image(alt_paths[0]))

## 📄 Step 9: Compile PDF Book
Creates a single PDF from all generated pages

In [ ]:
style = "sdxl_lora_grid" if USE_LORA else "sdxl_base_grid"
result = subprocess.run(
    [sys.executable, "compile_comic_pdf.py", "--style", style],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode == 0:
    pdf_path = f"outputs/comics/comic_book_{style}.pdf"
    if os.path.exists(pdf_path):
        size_mb = os.path.getsize(pdf_path) / 1024**2
        print(f"✅ PDF created: {pdf_path} ({size_mb:.1f} MB)")
else:
    print(f"Error: {result.stderr}")

## 📥 Step 10: Download All Outputs
Packages everything into a ZIP file for download

In [ ]:
import zipfile
import os

# Detect if running in Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CHARACTER_NAME = globals().get('CHARACTER_NAME', 'Wanderer')
STORY_WORLD = globals().get('STORY_WORLD', 'The_Abstract')

ZIP_NAME = f"comic_{CHARACTER_NAME.replace(' ', '_')}_{STORY_WORLD.replace(' ', '_')}.zip"
ZIP_PATH = f"/content/{ZIP_NAME}" if IN_COLAB else ZIP_NAME

print(f"📦 Creating {ZIP_NAME}...")

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk("outputs"):
        for file in files_list:
            if file.endswith(('.png', '.pdf', '.json')):
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, os.path.dirname("outputs"))
                zf.write(file_path, arcname)

size_mb = os.path.getsize(ZIP_PATH) / 1024**2
print(f"✅ ZIP created: {ZIP_NAME} ({size_mb:.1f} MB)")

if IN_COLAB:
    print("⬇️ Downloading...")
    files.download(ZIP_PATH)
    print("✅ Complete!")
else:
    print(f" Saved zip archive locally at: {os.path.abspath(ZIP_PATH)}")
    print(" Outputs are in the 'outputs/' directory.")

## ✨ Pipeline Complete!

### Summary:
- **Character**: {CHARACTER_NAME}
- **World**: {STORY_WORLD}
- **Pages Generated**: {NUM_PAGES}
- **Model**: SDXL + LoRA (Manga/Lineart style)
- **Resolution**: {IMG_WIDTH}x{IMG_HEIGHT}

### Output Files:
- Individual panels in `outputs/comics/`
- Character reference in `outputs/characters/`
- Storyboard JSON in `outputs/fusion/`
- Compiled PDF in `outputs/comics/`

### Next Time:
1. Re-run this notebook
2. Change CHARACTER_NAME and STORY_WORLD
3. Models will reload (takes 30-60 seconds first page)

**Enjoy your comic!** 🎉